In [ ]:
import torch
import pandas as pd
from transformers import pipeline
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

has_mps = torch.backends.mps.is_available()
device = "mps" if has_mps else "cpu"
pipeline_device = "mps" if has_mps else -1

print(f"Using device: {device}")

In [ ]:
model_name = "textattack/distilbert-base-uncased-MRPC"
clf = pipeline(
    "text-classification",
    model=model_name,
    tokenizer=model_name,
    device=pipeline_device,
    truncation=True,
    padding=True
)

print(f"Loaded pipeline model: {model_name}")
print(f"Pipeline framework device setting: {pipeline_device}")

In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation[128:256]")

print("Dataset split: glue/mrpc validation[128:256]")
print(f"Number of examples: {len(dataset)}")
print("Example row:")
print(dataset[0])

In [ ]:
batch_size = 32
paired_texts = [
    {"text": s1, "text_pair": s2}
    for s1, s2 in zip(dataset["sentence1"], dataset["sentence2"])
]

raw_outputs = clf(paired_texts, batch_size=batch_size)

labels = dataset["label"]
predictions = []
confidences = []

def normalize_label(label_text):
    t = str(label_text).strip().upper()
    if t.endswith("1"):
        return 1
    if t.endswith("0"):
        return 0
    raise ValueError(f"Unexpected label text: {label_text}")

for out in raw_outputs:
    pred = normalize_label(out["label"])
    predictions.append(pred)
    confidences.append(float(out["score"]))

print(f"Completed inference for {len(predictions)} examples.")

In [ ]:
accuracy = accuracy_score(labels, predictions)
f1 = f1_score(labels, predictions)
cm = confusion_matrix(labels, predictions)

print("Evaluation metrics on fixed subset validation[128:256]:")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1      : {f1:.4f}")
print("Confusion matrix:")
print(cm)
print("Classification report:")
print(classification_report(labels, predictions, target_names=["not_paraphrase", "paraphrase"], zero_division=0))

In [ ]:
rows = []
for i, row in enumerate(dataset):
    rows.append({
        "index": i,
        "true_label": labels[i],
        "pred_label": predictions[i],
        "confidence": confidences[i],
        "correct": int(labels[i] == predictions[i]),
        "sentence1": row["sentence1"],
        "sentence2": row["sentence2"],
    })

df = pd.DataFrame(rows).sort_values(by=["confidence", "index"], ascending=[True, True]).reset_index(drop=True)

display_cols = ["index", "true_label", "pred_label", "confidence", "correct", "sentence1", "sentence2"]
pd.set_option("display.max_colwidth", 120)
print("Per-example prediction table sorted by lowest confidence (first 20 rows):")
print(df[display_cols].head(20).to_string(index=False))

In [ ]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("dataset_split=glue/mrpc validation[128:256]")
print(f"device={device}")
print(f"num_examples={len(dataset)}")
print(f"accuracy={accuracy:.4f}")
print(f"f1={f1:.4f}")
print(f"confusion_matrix={cm.tolist()}")
print("lowest_confidence_examples=")
print(df[["index", "true_label", "pred_label", "confidence", "correct"]].head(10).to_string(index=False))